# PrefixSpan vs SPADE — synthetic scaling test

Both miners are exact: same input, same patterns, same supports (the benchmark asserts it).
They differ only in data structure — PrefixSpan re-scans a shrinking projected database,
SPADE joins vertical id-lists inside prefix equivalence classes.

This notebook is self-contained (no cohort needed): it doubles the number of synthetic
sequences each step and times both. It was moved out of Day 3, which uses PrefixSpan only —
on these tests PrefixSpan was faster at every size.


In [ ]:
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 150)


## PrefixSpan


In [ ]:
import collections

def prefixspan(db, min_sup, max_len=4):
    """Minimal PrefixSpan over single-item sequences.

    db      : list of lists of tokens
    min_sup : minimum number of sequences containing the pattern
    returns : DataFrame of (pattern, support, support_pct)
    """
    n, results = len(db), []

    def scan(projected, prefix):
        counts = collections.Counter()
        for seq in projected:
            for item in set(seq):                   # count each sequence once
                counts[item] += 1
        for item, c in counts.items():
            if c < min_sup:
                continue
            new_prefix = prefix + [item]
            results.append((tuple(new_prefix), c))
            if len(new_prefix) >= max_len:
                continue
            # project: keep the suffix after the FIRST occurrence of item
            nxt = [seq[seq.index(item) + 1:] for seq in projected if item in seq]
            if nxt:
                scan(nxt, new_prefix)

    scan([list(s) for s in db], [])
    return (pd.DataFrame(results, columns=['pattern', 'support'])
              .assign(support_pct=lambda d: (d.support / n * 100).round(1)))

## SPADE

PrefixSpan (above) mines **horizontally**: it re-scans a shrinking projected database at
every recursion step. SPADE (Zaki, 2001) mines **vertically**: build one id-list per item up
front — `item -> {sequence_id: positions}` — then grow patterns by *joining* id-lists
instead of touching the raw sequences again.

The pruning is the **equivalence class**: all frequent patterns sharing a prefix `P` form the
class `[P]`. A child of `P -> a` is only ever built by joining it with its siblings in `[P]`,
because `P -> a -> b` can only be frequent if `P -> b` is (Apriori). Candidates outside the
class are never tried.

Same rule as PrefixSpan for what counts as "next": the extension item must occur after the
prefix's *first* occurrence in that sequence. Same rule in, same patterns and supports out —
only the data structure and the traversal differ. That equivalence is what the scaling test
below asserts.

In [ ]:
import bisect

def spade(db, min_sup, max_len=4):
    """SPADE: vertical id-lists + temporal joins inside prefix equivalence classes.

    Same contract as prefixspan() -- same input, same output schema -- so the two
    are directly comparable.

    A class [P] holds the frequent atoms P->a. Children of P->a are built only by
    joining P->a with its *siblings* in [P]. That is the Apriori prune: P->a->b can
    only be frequent if P->b is, so any b outside [P] is never even tried.
    """
    n, results = len(db), []

    # vertical id-lists: occ[item][seq_id] = sorted positions of item in that sequence
    occ = collections.defaultdict(lambda: collections.defaultdict(list))
    for sid, seq in enumerate(db):
        for pos, item in enumerate(seq):
            occ[item][sid].append(pos)

    def join(idlist, item):
        """Temporal join: {sid: end pos of prefix} x item -> {sid: first pos of item after it}."""
        item_occ, out = occ[item], {}
        for sid, last in idlist.items():
            idxs = item_occ.get(sid)
            if idxs:
                k = bisect.bisect_right(idxs, last)
                if k < len(idxs):
                    out[sid] = idxs[k]
        return out

    def enumerate_class(prefix, atoms):
        # atoms: [(item, idlist)] -- frequent one-item extensions of `prefix`
        for item, idl in atoms:
            new_prefix = prefix + [item]
            results.append((tuple(new_prefix), len(idl)))
            if len(new_prefix) >= max_len:
                continue
            child = []
            for sib, _ in atoms:                    # equivalence-class prune: siblings only
                j = join(idl, sib)
                if len(j) >= min_sup:
                    child.append((sib, j))
            if child:
                enumerate_class(new_prefix, child)

    # depth-1 class: frequent items, each with its first occurrence per sequence
    root = [(item, {sid: idxs[0] for sid, idxs in item_occ.items()})
            for item, item_occ in occ.items()]
    root = [(item, idl) for item, idl in root if len(idl) >= min_sup]
    enumerate_class([], root)

    return (pd.DataFrame(results, columns=['pattern', 'support'])
              .assign(support_pct=lambda d: (d.support / n * 100).round(1)))


## Scaling test

Zipf-distributed vocabulary (a few ambient codes, a long tail), like real claims. Support is
fixed at 4% so pattern count stays roughly constant and the curves isolate database size.
The last two lines print the empirical growth exponent: slope of log time vs log n.


In [ ]:
def norm(d):
    return d.set_index('pattern').support.sort_index()

# Scaling test on synthetic data: double the number of sequences each step, time both miners.
# Zipf-distributed vocabulary (a few ambient codes, a long tail) like real claims; support is
# a fixed 4% so pattern count stays roughly constant and the curves isolate database size.
rng   = np.random.default_rng(0)
VOCAB = [f"c{i}" for i in range(60)]
w     = 1 / np.arange(1, len(VOCAB) + 1) ** 0.8; w /= w.sum()

def synth_db(n_seqs, length=30):
    return [list(rng.choice(VOCAB, size=length, p=w)) for _ in range(n_seqs)]

rows = []
for k in range(7, 18):                               # 256 ... 8192 sequences (~1 min total)
    n_seqs, sdb = 2 ** k, synth_db(2 ** k)
    ms = int(0.04 * n_seqs)
    t0 = time.perf_counter(); a = prefixspan(sdb, ms, max_len=4); t_a = time.perf_counter() - t0
    t0 = time.perf_counter(); b = spade(sdb, ms, max_len=4);      t_b = time.perf_counter() - t0
    assert norm(a).equals(norm(b)), f"miners disagree at n={n_seqs}"
    rows.append((n_seqs, len(a), t_a, t_b))
    print(f"n={n_seqs:>6}  patterns={len(a):>5}  prefixspan={t_a:7.2f}s  spade={t_b:7.2f}s  ratio={t_a / t_b:4.2f}")

bench = pd.DataFrame(rows, columns=['n_seqs', 'n_patterns', 'prefixspan_s', 'spade_s'])
ax = bench.plot(x='n_seqs', y=['prefixspan_s', 'spade_s'], marker='o', logx=True, logy=True, figsize=(6, 3.5))
ax.set_xlabel('sequences (doubling)'); ax.set_ylabel('seconds'); ax.set_title('PrefixSpan vs SPADE, synthetic'); plt.tight_layout(); plt.show()

# empirical growth exponent: slope of log(time) vs log(n). 1 = linear, 2 = quadratic.
for col in ['prefixspan_s', 'spade_s']:
    slope = np.polyfit(np.log(bench.n_seqs), np.log(bench[col]), 1)[0]
    print(f"{col:>13}: time ~ n^{slope:.2f}")
print(f"PrefixSpan faster at every size: {(bench.prefixspan_s < bench.spade_s).all()}"
      f"   (spade/prefixspan ratio {(bench.spade_s / bench.prefixspan_s).min():.1f}x-{(bench.spade_s / bench.prefixspan_s).max():.1f}x)")
